In [4]:
!pip install -q chromadb langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 13.2 MB/s eta 0:00:0000:01


In [5]:
import os
import json
from pathlib import Path
from getpass import getpass
from typing import TypedDict, Dict, Any, List, Literal

import pandas as pd
import chromadb
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

### Set models and storage config

In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_PATH = "career_chroma_db"
COLLECTION_NAME = "career_multi_agent_kb"
TOP_K = 5


embed_model = SentenceTransformer(EMBED_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
USER_PROFILE = {
    "name": "Md Mohsin",
    "target_role": "AI Engineer Intern",
    "experience_level": "student",
    "education": "CSE student",
    "current_skills": [
        "Python",
        "Machine Learning",
        "Deep Learning",
        "NLP",
        "Computer Vision",
        "scikit-learn"
    ],
    "tools_and_frameworks": [
        "PyTorch",
        "Transformers",
        "Flask",
        "Django",
        "Git",
        "Pandas"
    ],
    "projects_done": [
        "Spam detection NLP project",
        "E-commerce backend API project",
        "Vision transformer fine-tuning practice"
    ],
    "cv_text": """
Md Mohsin
CSE student interested in AI and backend development.
Worked on spam detection using machine learning and Flask.
Built an e-commerce backend API using Django REST Framework.
Learned ANN, CNN, RNN and worked on transformer fine-tuning practice.
Interested in AI engineer internship opportunities.
""",
    "target_job_description": """
We are hiring an AI Engineer Intern.
Requirements include Python, data preprocessing, model evaluation, machine learning fundamentals,
deep learning, NLP or computer vision exposure, Git, SQL basics, API integration,
clear communication, experimentation mindset, and ability to build end-to-end AI projects.
Experience with LLMs, vector databases, deployment, and notebooks is a plus.
""",
    "weekly_hours_available": 15,
    "notes": "I want a fresher-friendly roadmap and project ideas that improve my internship chances."
}

### Define structured schemas

In [8]:
class UserProfile(BaseModel):
    name: str
    target_role: str
    experience_level: Literal["student", "fresher", "junior", "switcher"] = "fresher"
    education: str = ""
    current_skills: List[str] = Field(default_factory=list)
    tools_and_frameworks: List[str] = Field(default_factory=list)
    projects_done: List[str] = Field(default_factory=list)
    cv_text: str
    target_job_description: str = ""
    weekly_hours_available: int = 12
    notes: str = ""

class CVReviewOutput(BaseModel):
    strengths: List[str]
    cv_improvement_points: List[str]
    rewritten_professional_summary: str
    keyword_gaps: List[str]

class SkillsGapOutput(BaseModel):
    role_fit_summary: str
    existing_strengths: List[str]
    missing_skills: List[str]
    missing_tools: List[str]
    learning_priority_order: List[str]

In [9]:
class ProjectIdea(BaseModel):
    title: str
    goal: str
    tech_stack: List[str]
    deliverables: List[str]
    why_this_project: str

class ProjectSuggestionsOutput(BaseModel):
    recommended_projects: List[ProjectIdea]

class InterviewTopic(BaseModel):
    topic: str
    why_it_matters: str
    sample_questions: List[str]

In [10]:
class InterviewPlanOutput(BaseModel):
    priority_topics: List[InterviewTopic]
    weekly_practice_strategy: str

class WeekPlan(BaseModel):
    week: int
    focus: str
    tasks: List[str]
    deliverable: str

class RoadmapOutput(BaseModel):
    thirty_day_goal: str
    weekly_plan: List[WeekPlan]
    checkpoint_metrics: List[str]

class FinalCareerPlan(BaseModel):
    coordinator_summary: str
    cv_improvement_points: List[str]
    missing_skills: List[str]
    project_recommendations: List[ProjectIdea]
    interview_prep_roadmap: List[InterviewTopic]
    thirty_day_plan: List[WeekPlan]
    next_7_days: List[str]

### Create helper functions

In [11]:
def clean_text(text):
    return str(text).strip()

def clean_list(items):
    if isinstance(items, list):
        return [str(x).strip() for x in items if str(x).strip()]
    if isinstance(items, str):
        return [x.strip() for x in items.split("\n") if x.strip()]
    return []

def chunk_text(text, chunk_size=900, overlap=120):
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap
    return chunks

def normalize_profile(data):
    profile = UserProfile(
        name=clean_text(data.get("name", "")),
        target_role=clean_text(data.get("target_role", "")),
        experience_level=clean_text(data.get("experience_level", "fresher")) or "fresher",
        education=clean_text(data.get("education", "")),
        current_skills=clean_list(data.get("current_skills", [])),
        tools_and_frameworks=clean_list(data.get("tools_and_frameworks", [])),
        projects_done=clean_list(data.get("projects_done", [])),
        cv_text=clean_text(data.get("cv_text", "")),
        target_job_description=clean_text(data.get("target_job_description", "")),
        weekly_hours_available=int(data.get("weekly_hours_available", 12)),
        notes=clean_text(data.get("notes", ""))
    )
    return profile.model_dump()

def retrieve_docs(query, top_k=TOP_K):
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    query_embedding = embed_model.encode([query], normalize_embeddings=True).tolist()[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)

    parsed = []
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    for doc, meta, dist in zip(documents, metadatas, distances):
        parsed.append(
            {
                "title": meta.get("title", ""),
                "kind": meta.get("kind", ""),
                "source": meta.get("source", ""),
                "text": doc,
                "distance": dist
            }
        )
    return parsed

def format_context(items):
    lines = []
    for i, item in enumerate(items, start=1):
        lines.append(
            f"[{i}] Title: {item['title']}\nKind: {item['kind']}\nSource: {item['source']}\nContent: {item['text']}"
        )
    return "\n\n".join(lines)

def source_titles(items):
    return [f"{item['title']} ({item['kind']}, {item['source']})" for item in items]

def make_projects_df(projects):
    rows = []
    for item in projects:
        rows.append(
            {
                "title": item["title"],
                "goal": item["goal"],
                "tech_stack": ", ".join(item["tech_stack"]),
                "deliverables": " | ".join(item["deliverables"]),
                "why_this_project": item["why_this_project"]
            }
        )
    return pd.DataFrame(rows)

def make_interview_df(topics):
    rows = []
    for item in topics:
        rows.append(
            {
                "topic": item["topic"],
                "why_it_matters": item["why_it_matters"],
                "sample_questions": " | ".join(item["sample_questions"])
            }
        )
    return pd.DataFrame(rows)

def make_roadmap_df(weeks):
    rows = []
    for item in weeks:
        rows.append(
            {
                "week": item["week"],
                "focus": item["focus"],
                "tasks": " | ".join(item["tasks"]),
                "deliverable": item["deliverable"]
            }
        )
    return pd.DataFrame(rows)

def make_retrieval_df(retrieval_log):
    rows = []
    for agent_name, sources in retrieval_log.items():
        for source in sources:
            rows.append(
                {
                    "agent": agent_name,
                    "source": source
                }
            )
    return pd.DataFrame(rows)

def build_markdown_report(final_output):
    lines = []
    lines.append("# Multi-Agent Career Assistant Report")
    lines.append("")
    lines.append("## Coordinator Summary")
    lines.append(final_output["coordinator_summary"])
    lines.append("")
    lines.append("## CV Improvement Points")
    for item in final_output["cv_improvement_points"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Missing Skills")
    for item in final_output["missing_skills"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Project Recommendations")
    for item in final_output["project_recommendations"]:
        lines.append(f"### {item['title']}")
        lines.append(f"**Goal:** {item['goal']}")
        lines.append(f"**Tech Stack:** {', '.join(item['tech_stack'])}")
        lines.append(f"**Deliverables:** {' | '.join(item['deliverables'])}")
        lines.append(f"**Why It Matters:** {item['why_this_project']}")
        lines.append("")
    lines.append("## Interview Prep Roadmap")
    for item in final_output["interview_prep_roadmap"]:
        lines.append(f"### {item['topic']}")
        lines.append(item["why_it_matters"])
        for q in item["sample_questions"]:
            lines.append(f"- {q}")
        lines.append("")
    lines.append("## 30-Day Plan")
    for item in final_output["thirty_day_plan"]:
        lines.append(f"### Week {item['week']} — {item['focus']}")
        for task in item["tasks"]:
            lines.append(f"- {task}")
        lines.append(f"**Deliverable:** {item['deliverable']}")
        lines.append("")
    lines.append("## Next 7 Days")
    for item in final_output["next_7_days"]:
        lines.append(f"- {item}")
    return "\n".join(lines)

### Create the career knowledge base

In [12]:
def build_knowledge_base_docs():
    docs = [
        {
            "id": "kb_resume_1",
            "title": "Fresher CV basics",
            "kind": "resume",
            "source": "knowledge_base",
            "text": "A fresher CV should highlight target role alignment, strong summary, core skills, relevant tools, academic or personal projects, quantified impact, GitHub or portfolio links, and concise formatting."
        },
        {
            "id": "kb_resume_2",
            "title": "Project bullet writing",
            "kind": "resume",
            "source": "knowledge_base",
            "text": "Strong project bullets should mention the problem, dataset or input, method used, stack, evaluation metric, deployment or demo, and business or user impact. Generic bullets should be rewritten into measurable evidence."
        },
        {
            "id": "kb_resume_3",
            "title": "ATS keyword strategy",
            "kind": "resume",
            "source": "knowledge_base",
            "text": "For internship and fresher hiring, CVs improve when role keywords from the job description are mirrored naturally in skills, summary, projects, and tools. Important keywords often include Python, SQL, APIs, Git, experimentation, model evaluation, and deployment basics."
        },
        {
            "id": "kb_role_ai_1",
            "title": "AI engineer role expectations",
            "kind": "role",
            "source": "knowledge_base",
            "text": "AI engineer and ML internship roles usually expect Python, data preprocessing, model training, evaluation metrics, notebook workflows, Git, APIs, and at least two end-to-end projects in NLP, computer vision, tabular ML, or LLM applications."
        },
        {
            "id": "kb_role_ai_2",
            "title": "AI engineer bonus skills",
            "kind": "role",
            "source": "knowledge_base",
            "text": "Bonus skills for AI engineer roles often include SQL, deployment basics, Docker, vector databases, retrieval augmented generation, experiment tracking, cloud familiarity, and ability to explain tradeoffs between models."
        },
        {
            "id": "kb_role_ai_3",
            "title": "AI engineer portfolio direction",
            "kind": "role",
            "source": "knowledge_base",
            "text": "A strong AI portfolio usually includes one classical ML project, one deep learning or transformer project, and one production-style project such as an API, dashboard, or assistant. Recruiters value clarity, metrics, and practical business framing."
        },
        {
            "id": "kb_role_data_1",
            "title": "Data analyst role expectations",
            "kind": "role",
            "source": "knowledge_base",
            "text": "Data analyst roles typically expect SQL, Excel or spreadsheets, pandas, visualization, descriptive statistics, dashboarding, business communication, and project examples involving trends, KPIs, and stakeholder-friendly reporting."
        },
        {
            "id": "kb_role_backend_1",
            "title": "Backend developer role expectations",
            "kind": "role",
            "source": "knowledge_base",
            "text": "Backend roles typically expect APIs, databases, authentication, Git, testing, clean architecture, deployment familiarity, and at least two hands-on backend projects."
        },
        {
            "id": "kb_project_1",
            "title": "Project idea - RAG document assistant",
            "kind": "project",
            "source": "knowledge_base",
            "text": "A strong AI project is a document QA assistant that loads PDFs, chunks text, creates embeddings, performs vector search, and answers with citations. It demonstrates NLP, retrieval, prompting, evaluation, and product thinking."
        },
        {
            "id": "kb_project_2",
            "title": "Project idea - NLP classifier",
            "kind": "project",
            "source": "knowledge_base",
            "text": "A useful fresher project is an NLP classifier such as spam detection, support ticket classification, or sentiment analysis with a clean training pipeline, metrics, confusion analysis, and a small app or API demo."
        },
        {
            "id": "kb_project_3",
            "title": "Project idea - Computer vision system",
            "kind": "project",
            "source": "knowledge_base",
            "text": "A computer vision project becomes stronger when it includes data preprocessing, augmentation, evaluation, error analysis, and a realistic use case such as defect detection, attendance, or document analysis."
        },
        {
            "id": "kb_project_4",
            "title": "Project idea - Data analysis agent",
            "kind": "project",
            "source": "knowledge_base",
            "text": "An analysis assistant that reads CSV files, profiles schema, finds missing values, suggests charts, and explains insights is useful for both analytics and AI portfolios because it combines data handling, reasoning, and automation."
        },
        {
            "id": "kb_project_5",
            "title": "Project idea - Multi-agent assistant",
            "kind": "project",
            "source": "knowledge_base",
            "text": "A multi-agent assistant demonstrates decomposition, coordination, role-specific prompting, retrieval, and structured outputs. It is especially good for LLM and agent engineering portfolios."
        },
        {
            "id": "kb_interview_1",
            "title": "AI interview core topics",
            "kind": "interview",
            "source": "knowledge_base",
            "text": "AI engineer interviews commonly test data preprocessing, train validation test split, overfitting, bias variance, evaluation metrics, embeddings, transformers, optimization basics, and tradeoffs between classical ML and deep learning."
        },
        {
            "id": "kb_interview_2",
            "title": "AI interview project discussion",
            "kind": "interview",
            "source": "knowledge_base",
            "text": "Candidates should be ready to explain their project problem statement, dataset, model choice, metrics, errors, limitations, and what they would improve next. Interviewers often care more about reasoning than tool names."
        },
        {
            "id": "kb_interview_3",
            "title": "Behavioral interview basics",
            "kind": "interview",
            "source": "knowledge_base",
            "text": "Freshers should prepare short stories around teamwork, learning speed, problem solving, handling deadlines, project ownership, and why they fit the target role."
        },
        {
            "id": "kb_roadmap_1",
            "title": "Four-week career sprint",
            "kind": "roadmap",
            "source": "knowledge_base",
            "text": "A four-week career sprint works well when week one focuses on CV and gaps, week two on skill strengthening, week three on one flagship project, and week four on interview practice plus applications."
        },
        {
            "id": "kb_roadmap_2",
            "title": "Study planning principles",
            "kind": "roadmap",
            "source": "knowledge_base",
            "text": "A realistic plan should match weekly time availability, limit concurrent goals, include visible deliverables, and end each week with a measurable checkpoint such as a project demo, revised CV, or completed mock interview."
        }
    ]
    return docs

def build_user_docs(profile):
    docs = []

    docs.append(
        {
            "id": "user_profile_summary",
            "title": "User profile summary",
            "kind": "user_profile",
            "source": "user_input",
            "text": f"Name: {profile['name']}. Target role: {profile['target_role']}. Experience level: {profile['experience_level']}. Education: {profile['education']}. Current skills: {', '.join(profile['current_skills'])}. Tools: {', '.join(profile['tools_and_frameworks'])}. Projects done: {', '.join(profile['projects_done'])}. Weekly hours: {profile['weekly_hours_available']}. Notes: {profile['notes']}"
        }
    )

    cv_chunks = chunk_text(profile["cv_text"])
    for i, chunk in enumerate(cv_chunks, start=1):
        docs.append(
            {
                "id": f"user_cv_{i}",
                "title": f"User CV chunk {i}",
                "kind": "cv",
                "source": "user_input",
                "text": chunk
            }
        )

    if profile["target_job_description"]:
        jd_chunks = chunk_text(profile["target_job_description"])
        for i, chunk in enumerate(jd_chunks, start=1):
            docs.append(
                {
                    "id": f"user_jd_{i}",
                    "title": f"Target job description chunk {i}",
                    "kind": "job_description",
                    "source": "user_input",
                    "text": chunk
                }
            )

    return docs

def setup_collection(profile):
    all_docs = build_knowledge_base_docs() + build_user_docs(profile)

    try:
        chroma_client.delete_collection(name=COLLECTION_NAME)
    except:
        pass

    collection = chroma_client.create_collection(name=COLLECTION_NAME)

    ids = [item["id"] for item in all_docs]
    texts = [item["text"] for item in all_docs]
    metadatas = [
        {
            "title": item["title"],
            "kind": item["kind"],
            "source": item["source"]
        }
        for item in all_docs
    ]
    embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()

    collection.add(
        ids=ids,
        documents=texts,
        metadatas=metadatas,
        embeddings=embeddings
    )

    return {
        "collection_name": COLLECTION_NAME,
        "document_count": len(all_docs)
    }

### Define workflow state

In [ ]:
class CareerState(TypedDict):
    raw_input: Dict[str, Any]
    profile: Dict[str, Any]
    vector_store_info: Dict[str, Any]
    retrieval_log: Dict[str, Any]
    cv_review: Dict[str, Any]
    skills_gap: Dict[str, Any]
    projects: Dict[str, Any]
    interview_plan: Dict[str, Any]
    roadmap: Dict[str, Any]
    final_output: Dict[str, Any]
    report_markdown: str